# §13.12.5 — 구조를 입력하면 예산을 출력하는 계산기

> 딥러닝 교재 · 3부 13장 12절 5항 (🐍)
> 선행: §13.12.1(파라미터 공식) · §13.12.2(6ND) · §13.12.3(KV 캐시) · §13.12.6(작은 항)

## 이 노트북이 답하는 질문

1. **공식 $12Ld^2+V_{\rm oc}d$는 실제 공개 모델을 몇 % 오차로 재현하는가?**
2. **규모에 따라 파라미터 구성비는 어떻게 이동하는가?**
3. **KV 캐시와 학습 연산량은 설계 선택(GQA, 문맥, 배치)에 어떻게 반응하는가?**

**예상 실행 시간** CPU 약 10초.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 계산기 — 세 공식의 조립

입력: $(d, L, h, h_{kv}, d_{\rm ff}, V_{\rm oc}, T, B)$와 MLP 종류.
출력: 파라미터 분해 · 학습 FLOPs($6ND$) · KV 캐시 · 활성 메모리 개산.
게이트형 MLP(SwiGLU류)는 행렬이 3개라 $3\,d\,d_{\rm ff}$로 센다.

In [ ]:
def budget(d, L, h, V_oc, d_ff=None, h_kv=None, mlp='gelu', T=2048, B=1,
           bytes_per=2, tied=True):
    d_ff = d_ff or 4 * d
    h_kv = h_kv or h
    d_h = d // h
    attn = L * (2 * d * d + 2 * d * h_kv * d_h)               # Wq,Wo는 d², Wk,Wv는 h_kv 몫
    mlp_p = L * (3 if mlp == 'gated' else 2) * d * d_ff
    emb = V_oc * d * (1 if tied else 2)
    small = L * 4 * d + 2 * d                                  # 정규화 이득·편향 근사
    N = attn + mlp_p + emb + small
    flops_per_tok = 6 * N                                      # 학습 (§13.12.2)
    kv = 2 * L * h_kv * d_h * T * B * bytes_per                # §13.12.3
    act = B * T * d * L * 6 * bytes_per                        # 활성 개산 (블록당 상수 6)
    return dict(N=N, attn=attn, mlp=mlp_p, emb=emb, small=small,
                flops_per_tok=flops_per_tok, kv=kv, act=act)

MODELS = {
    'GPT-2 124M':  dict(d=768,  L=12, h=12, V_oc=50257, mlp='gelu', rep=124e6),
    'GPT-2 355M':  dict(d=1024, L=24, h=16, V_oc=50257, mlp='gelu', rep=355e6),
    'GPT-2 774M':  dict(d=1280, L=36, h=20, V_oc=50257, mlp='gelu', rep=774e6),
    'GPT-2 1.5B':  dict(d=1600, L=48, h=25, V_oc=50257, mlp='gelu', rep=1558e6),
    'GPT-3 175B':  dict(d=12288, L=96, h=96, V_oc=50257, mlp='gelu', rep=175e9),
    'LLaMA 7B':    dict(d=4096, L=32, h=32, V_oc=32000, d_ff=11008, mlp='gated',
                        tied=False, rep=6.74e9),
    'LLaMA-2 70B': dict(d=8192, L=80, h=64, h_kv=8, V_oc=32000, d_ff=28672,
                        mlp='gated', tied=False, rep=69e9),
}
print(f"{'모델':14s} {'공식':>9s} {'보고치':>9s} {'오차':>7s}")
errs = {}
for nm, cfgm in MODELS.items():
    rep = cfgm.pop('rep')
    b = budget(**cfgm)
    err = (b['N'] - rep) / rep * 100
    errs[nm] = (b, rep, err)
    cfgm['rep'] = rep
    print(f"{nm:14s} {b['N']/1e9:8.2f}B {rep/1e9:8.2f}B {err:+6.1f}%")

---
## 2. 구성비의 이동, 캐시, 연산량 지도

In [ ]:
# 규모 사다리: GPT-2 계열의 관행(V 고정, L∝d)을 따라 d를 훑는다
ds = np.array([256, 512, 768, 1024, 1600, 2560, 4096, 8192, 12288])
Ls = np.maximum(4, (ds / 64).astype(int))
comp = np.array([[budget(d=int(dd), L=int(LL), h=max(4, int(dd) // 64), V_oc=50257)[k]
                  for k in ['attn', 'mlp', 'emb', 'small']] for dd, LL in zip(ds, Ls)], float)
comp_share = comp / comp.sum(1, keepdims=True)

# KV 캐시: 7B급 구성에서 문맥을 훑으며 h_kv 비교
Ts = np.array([1024, 4096, 16384, 65536])
kv_curves = {}
for nm, hkv in [('MHA ($h_{kv}=32$)', 32), ('GQA ($h_{kv}=8$)', 8), ('MQA ($h_{kv}=1$)', 1)]:
    kv_curves[nm] = [budget(d=4096, L=32, h=32, h_kv=hkv, V_oc=32000,
                            d_ff=11008, mlp='gated', T=int(t))['kv'] / 2**30 for t in Ts]

# 6ND 지도
N_grid = np.logspace(7, 12, 60)
D_grid = np.logspace(8, 13, 60)
NN, DD = np.meshgrid(N_grid, D_grid)
CC = 6 * NN * DD
print("준비 완료")

---
## 3. 교재 그림 — fig_13_12_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 공식 재현 오차
ax = axes[0]
names = list(errs.keys())
vals = [errs[nm][2] for nm in names]
ax.barh(range(len(names)), vals, color=[CB[5] if abs(v) < 5 else CB[1] for v in vals])
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel(lab('공식 대 보고치 오차 (%)', 'error vs reported (%)'))
ax.set_title(lab('(a) 봉투 뒷면 공식의 재현 오차', '(a) formula vs reported'), fontsize=10)

# (b) 구성비의 이동
ax = axes[1]
labels_b = [lab('어텐션', 'attn'), 'MLP', lab('임베딩', 'emb'), lab('기타', 'small')]
ax.stackplot(ds, comp_share.T * 100, labels=labels_b,
             colors=[CB[5], CB[3], CB[1], CB[7]], alpha=0.85)
ax.set_xscale('log')
ax.set_xlabel(lab('모델 폭 $d$ (로그 축, $L\\propto d$)', 'width $d$'))
ax.set_ylabel(lab('파라미터 구성비 (%)', 'share (%)'))
ax.set_title(lab('(b) 소형에선 임베딩이, 대형에선 블록이 지배한다', '(b) composition'), fontsize=10)
ax.legend(fontsize=8, loc='center right')

# (c) KV 캐시
ax = axes[2]
for i, (nm, cur) in enumerate(kv_curves.items()):
    ax.loglog(Ts, cur, 'o-', color=[CB[5], CB[3], CB[1]][i], ms=5, label=nm)
ax.axhline(14, color='k', lw=0.8, ls=':')
ax.text(Ts[0], 15.5, lab('7B 가중치(fp16) ≈ 14 GiB', 'weights'), fontsize=8)
ax.set_xlabel(lab('문맥 길이 $T$', 'context length'))
ax.set_ylabel(lab('KV 캐시 (GiB, $B=1$, fp16)', 'KV cache (GiB)'))
ax.set_title(lab('(c) 캐시는 문맥에 선형 — 헤드 공유가 기울기를 낮춘다', '(c) KV cache'), fontsize=10)
ax.legend(fontsize=8)

# (d) 6ND 지도
ax = axes[3]
cs = ax.contour(NN, DD, np.log10(CC), levels=np.arange(17, 26, 1), cmap='viridis')
ax.clabel(cs, fmt=lambda v: f'$10^{{{v:.0f}}}$', fontsize=7)
pts = {'GPT-2': (124e6, 40e9), 'GPT-3': (175e9, 300e9), 'LLaMA 7B': (6.7e9, 1e12)}
for nm, (n_, d_) in pts.items():
    ax.plot(n_, d_, 'o', color=CB[4], ms=6)
    ax.annotate(nm, (n_, d_), xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(lab('파라미터 수 $N$', 'params $N$'))
ax.set_ylabel(lab('학습 토큰 수 $D$', 'tokens $D$'))
ax.set_title(lab('(d) $C=6ND$의 지도 — 등연산량 곡선', '(d) compute map'), fontsize=10)

save_book_fig(fig, 'fig_13_12_5')
plt.show()

> ### 읽는 법
>
> (a) 표준 구성(GPT-2·GPT-3)은 수 퍼센트 안에서 재현된다 — 공식이 아니라 구조가
> 단순한 덕이다. LLaMA류도 게이트형 MLP($3dd_{\rm ff}$)와 비묶음 임베딩만 반영하면
> 맞아떨어진다. 봉투 뒷면 계산의 신뢰 구간이 이 정도라는 것을 몸에 새겨 두자.
> (b) 같은 공식의 두 항이 규모에 따라 주역을 바꾼다 — 소형 모델에서 절반이던
> 임베딩 몫이 대형에서 반올림 오차가 된다(§13.12.6의 "작은 항" 경고).
> (c) KV 캐시는 문맥에 선형이고, 긴 문맥에서는 가중치보다 커진다. $h_{kv}$를 줄이는
> GQA/MQA가 곡선 전체를 끌어내리는 것이 식 13.12.3 그대로다.
> (d) $(N, D)$ 평면의 등연산량 곡선. 고정 예산 $C$에서 어디에 앉을 것인가 — 이 지도
> 위의 최적 자리를 찾는 것이 38장 스케일링 법칙의 문제다.

---
## 5. 자기 점검

1. (a)에서 GPT-2 124M의 오차를 0.5% 아래로 줄여 보라. 어떤 "작은 항"을 더 넣어야 하는가? (위치 임베딩 $T_{\max}d$를 잊지 않았는가?)
2. 어휘 25만의 다국어 모델을 $d=1024, L=24$로 만들면 임베딩 몫이 몇 %인가? 계산기로 확인하라.
3. (c)에서 $B=64$ 서빙이라면 GQA 7B의 32k 문맥 캐시는 몇 GiB인가? A100(80GiB) 몇 장인가?
4. $C=10^{23}$ FLOPs 예산에서 $N=10^{10}$을 고르면 $D$는 얼마인가? (d)의 지도에서 자리를 찾아보라.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `MODELS` | 1절 | 7개 | 아는 모델을 추가해 검증 |
| `bytes_per` | 1절 | 2 (fp16) | 양자화의 캐시 절감 |
| `tied` | 1절 | 모델별 | 임베딩 묶음의 효과 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")